# Actividad 15

- Descargue datos de cotización de cierre para MELI (symbol MELI.BA) y  YPF (symbol YPFD.BA) mediante la API de Yahoo Finance, para los dos últimos meses disponibles de forma mensual. La columna Date conviértala al nombre del mes. 


In [1]:

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import yfinance as yf

pio.renderers.default = 'plotly_mimetype'

# ── Descarga de datos ──────────────────────────────────────────────
# Se pide period='3mo' para asegurar obtener al menos 2 meses completos
# y luego quedarse con los 2 últimos disponibles.
def descargar_activo(ticker):
    df = (
        yf.download(ticker, period='3mo', interval='1mo')[['Close']]
        .reset_index()
    )
    df.columns = df.columns.droplevel(1)
    df = df.assign(Activo=ticker)
    return df

df_meli = descargar_activo('MELI.BA')
df_ypf  = descargar_activo('YPFD.BA')

# Unir y quedarse con los 2 últimos meses disponibles por activo
data = pd.concat([df_meli, df_ypf], ignore_index=True)
data['Date'] = pd.to_datetime(data['Date'])

# Ordenar y tomar los últimos 2 períodos por activo
data = (
    data.sort_values('Date')
    .groupby('Activo')
    .tail(2)
    .reset_index(drop=True)
)

# Convertir Date al nombre del mes
MESES_ES = {
    1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril',
    5:'Mayo',  6:'Junio',   7:'Julio', 8:'Agosto',
    9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'
}
data['Mes'] = data['Date'].dt.month.map(MESES_ES)

print(data[['Activo', 'Mes', 'Close']].to_string(index=False))


/tmp/ipykernel_129614/604834155.py:13: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_129614/604834155.py:13: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed

 Activo   Mes   Close
MELI.BA  Mayo 20930.0
YPFD.BA  Mayo 78350.0
MELI.BA Junio 20230.0
YPFD.BA Junio 81075.0


- Mostrar por pantalla los siguientes gráficos pedidos:


A) Pivotee el dataframe y calcule la variación mensual. Grafique con puntos conectados y agregue la variación
porcentual indicando si subió o bajó con flecha y valor


In [2]:

# A — Puntos conectados con variación porcentual y flechas

# Pivot: filas = Mes (en orden cronológico), columnas = Activo
pivot = data.pivot(index='Mes', columns='Activo', values='Close')

# Reordenar filas según fecha real (no alfabética)
orden_meses = data.drop_duplicates('Mes').sort_values('Date')['Mes'].tolist()
pivot = pivot.loc[orden_meses]

# Calcular variación porcentual entre los dos meses
variaciones = ((pivot.iloc[1] - pivot.iloc[0]) / pivot.iloc[0] * 100).round(2)

fig = go.Figure()

colores = {'MELI.BA': '#1f77b4', 'YPFD.BA': '#ff7f0e'}

for activo in pivot.columns:
    fig.add_trace(go.Scatter(
        x=pivot.index,
        y=pivot[activo],
        mode='lines+markers',
        name=activo,
        line=dict(color=colores.get(activo), width=3),
        marker=dict(size=12)
    ))

    var = variaciones[activo]
    flecha = '▲' if var > 0 else '▼'
    color_flecha = 'green' if var > 0 else 'red'

    # Anotación con flecha y valor sobre el segundo punto (mes más reciente)
    fig.add_annotation(
        x=pivot.index[1],
        y=pivot[activo].iloc[1],
        text=f"<b>{flecha} {abs(var):.1f}%</b>",
        showarrow=True,
        arrowhead=2,
        arrowcolor=color_flecha,
        font=dict(color=color_flecha, size=13),
        ax=40, ay=-35,
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor=color_flecha,
        borderwidth=1
    )

fig.update_layout(
    title=dict(
        text='Evolución mensual de cierre — MELI.BA y YPFD.BA',
        x=0.5, xanchor='center', font=dict(size=15)
    ),
    xaxis_title='Mes',
    yaxis_title='Precio de cierre ($)',
    yaxis=dict(tickformat='$,.0f'),
    legend=dict(title='Activo'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=450
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#eeeeee')
fig.show()


B) Realice un gráfico de indicador por cada activo para visualizar la variación ente los dos meses. Los gráficos 
deben visualizar en un mismo espacio, uno al lado del otro 


In [3]:

# B — Indicadores de variación, uno por activo, side-by-side

fig = go.Figure()

for col_idx, activo in enumerate(pivot.columns):
    precio_actual    = round(float(pivot[activo].iloc[1]), 2)
    precio_anterior  = round(float(pivot[activo].iloc[0]), 2)
    mes_actual       = pivot.index[1]
    mes_anterior     = pivot.index[0]

    fig.add_trace(go.Indicator(
        mode='number+delta',
        value=precio_actual,
        delta=dict(
            reference=precio_anterior,
            valueformat='.2f',
            relative=True,
            position='bottom'
        ),
        title=dict(
            text=f'<b>{activo}</b><br>'
                 f'<span style="font-size:0.85em;color:gray">'
                 f'Cierre {mes_actual} vs {mes_anterior}</span>',
            font=dict(size=16)
        ),
        number=dict(prefix='$', valueformat=',.2f'),
        domain=dict(row=0, column=col_idx)
    ))

fig.update_layout(
    grid=dict(rows=1, columns=len(pivot.columns), pattern='independent'),
    template=dict(data=dict(indicator=[dict(
        mode='number+delta',
        delta=dict(increasing=dict(color='green'), decreasing=dict(color='red'))
    )])),
    height=280,
    paper_bgcolor='white',
    title=dict(
        text='Variación mensual por activo',
        x=0.5, xanchor='center', font=dict(size=15)
    )
)
fig.show()


C) Realice un gráfico de coordenadas paralelas, con ejes numéricos y agregar etiquetas nombres activos

In [4]:

# C — Coordenadas paralelas: ejes numéricos (precios por mes) + etiquetas de activos

mes1 = orden_meses[0]   # primer mes
mes2 = orden_meses[1]   # segundo mes

# pivot ya tiene filas=Mes, cols=Activo → transponer para filas=Activo
df_parcoords = pivot.T.reset_index()
df_parcoords.columns = ['Activo', mes1, mes2]
df_parcoords['idx'] = df_parcoords.index   # índice numérico para colorear

# Valores del eje para mostrar etiquetas en el primer eje
tickvals1 = df_parcoords[mes1].tolist()
ticktext1 = df_parcoords['Activo'].tolist()

rng1 = [df_parcoords[mes1].min() * 0.98, df_parcoords[mes1].max() * 1.02]
rng2 = [df_parcoords[mes2].min() * 0.98, df_parcoords[mes2].max() * 1.02]

fig = go.Figure(data=go.Parcoords(
    line=dict(
        color=df_parcoords['idx'],
        colorscale='agsunset',
        showscale=False
    ),
    dimensions=[
        dict(
            range=rng1,
            label=mes1,
            values=df_parcoords[mes1],
            tickvals=tickvals1,
            ticktext=ticktext1
        ),
        dict(
            range=rng2,
            label=mes2,
            values=df_parcoords[mes2]
        ),
    ]
))

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    title=dict(
        text=f'Coordenadas paralelas — Precio de cierre {mes1} vs {mes2}',
        x=0.5, xanchor='center', font=dict(size=15)
    ),
    height=400
)
fig.show()
